# PromptEHR Synthetic Data Generation for MIMIC-III

_Last updated: 2026-03-05_

This notebook trains PromptEHR on your MIMIC-III data and generates synthetic patients conditioned on patient demographics (age and gender).

## What You'll Need

1. **MIMIC-III Access**: Download these files from PhysioNet:
   - `PATIENTS.csv` — patient demographics (date of birth, gender)
   - `ADMISSIONS.csv` — hospital admission records (timestamps)
   - `DIAGNOSES_ICD.csv` — ICD-9 diagnosis codes

2. **Google Colab**: Free tier works, but GPU recommended (Runtime → Change runtime type → GPU)

3. **Time**:
   - Demo (5 epochs, 1K samples): ~30–45 min on GPU
   - Production (20 epochs, 10K samples): ~3–5 hrs on GPU

## How It Works

1. **Setup**: Install PyHealth and mount Google Drive
2. **Upload Data**: Upload your MIMIC-III CSV files (persisted to Drive across sessions)
3. **Configure**: Set hyperparameters (epochs, batch size, etc.)
4. **Train**: Fine-tune PromptEHR on MIMIC-III (checkpoints saved to Drive)
5. **Generate**: Create synthetic patients conditioned on sampled demographics
6. **Download**: Get CSV file with synthetic data

## Important Notes

**Colab Timeout**: Free Colab sessions timeout after ~12 hours. For production training, consider:
- Colab Pro for longer sessions
- Running on a GPU cluster using `examples/slurm/`

**Demo vs Production**:
- Demo defaults (5 epochs, 1K samples) let you test the full pipeline quickly
- Production settings (20 epochs, 10K samples) produce publication-quality results

## References

- [PromptEHR Paper](https://arxiv.org/abs/2211.01761) — Wang et al., EMNLP 2023
- [PyHealth Documentation](https://pyhealth.readthedocs.io/)
- [MIMIC-III Access](https://physionet.org/content/mimiciii/)

---
# 1. Setup & Installation

In [ ]:
import subprocess
import sys

# Install PyHealth from GitHub.
# We uninstall first (to clear any stale build from a previous session),
# then do a normal install. We do NOT use --force-reinstall because it
# force-reinstalls ALL transitive deps, which in Colab's system environment
# creates mixed-version states (old .so + new .py from different versions).
FORK = 'jalengg'
BRANCH = 'promptehr-pr-integration'
install_url = f"git+https://github.com/{FORK}/PyHealth.git@{BRANCH}"

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "pyhealth", "-y"],
    capture_output=True, text=True,
)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", install_url,
     "--quiet", "--no-cache-dir"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("PyHealth installation failed — see error above.")
print(f"✓ PyHealth installed from {FORK}/{BRANCH}")

In [ ]:
import os
import random
import shutil
import numpy as np
import torch
import pandas as pd
from IPython.display import display
from google.colab import drive, files

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("  → Runtime → Change runtime type → T4 GPU")

# Verify HuggingFace transformers version (>=4.48.3 required for use_cpu parameter)
import transformers
_ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert _ver >= (4, 48), (
    f"transformers>=4.48.3 required (got {transformers.__version__}). "
    "Fix: pip install transformers --upgrade"
)
print(f"transformers: {transformers.__version__} ✓")

In [ ]:
# Mount Google Drive for persistent storage
print("Mounting Google Drive...")
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("Drive already mounted")
print("✓ Google Drive mounted")

# Create directory structure in Drive
BASE_DIR       = '/content/drive/MyDrive/PromptEHR_Training'
DATA_DIR       = f'{BASE_DIR}/data'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints'
OUTPUT_DIR     = f'{BASE_DIR}/output'

for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"\nDirectory structure created:")
print(f"  Base: {BASE_DIR}")
print(f"  Data: {DATA_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Output: {OUTPUT_DIR}")

---
# 2. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — All modifiable parameters in one place
# ============================================================

# --- Training parameters ---
EPOCHS              = 5       # Demo: 5, Production: 20
BATCH_SIZE          = 16      # 16 for both
N_SYNTHETIC_SAMPLES = 1_000   # Demo: 1000, Production: 10000
WARMUP_STEPS        = 100     # Demo: 100, Production: 1000

LR             = 1e-5   # Paper LR; low to avoid catastrophic forgetting of BART weights
MAX_SEQ_LENGTH = 512    # Max tokens per patient (visits + special tokens)

# --- Model architecture ---
D_HIDDEN      = 128  # Hidden dim for demographic prompt encoder
PROMPT_LENGTH = 1    # Prompt vectors per demographic feature (1 is sufficient per paper)

# --- BART backbone ---
# "facebook/bart-base": pretrained BART (139 M params, 768 hidden dim).
# PromptEHR fine-tunes these weights rather than training from scratch —
# the pretrained sequence modeling prior means even 20 epochs can produce good results.
BART_CONFIG_NAME = "facebook/bart-base"

# --- Generation parameters ---
RANDOM_SAMPLING = True   # True: nucleus sampling (diverse), False: greedy (deterministic)
TEMPERATURE     = 0.7    # Lower = more common codes. Higher = more rare/diverse codes.
TOP_P           = 0.95   # Nucleus sampling: sample from top 95% probability mass.

# --- Reproducibility ---
SEED = 42

# Display configuration
print("=" * 60)
print("PROMPTEHR CONFIGURATION")
print("=" * 60)
print(f"Training:")
print(f"  Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LR}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"\nGeneration:")
print(f"  Synthetic samples: {N_SYNTHETIC_SAMPLES:,}")
print(f"  Sampling: {'nucleus (random)' if RANDOM_SAMPLING else 'greedy'}")
print(f"\nPaths:")
print(f"  Base directory: {BASE_DIR}")
print("=" * 60)

---
# 3. Data Upload

Upload your MIMIC-III CSV files. PromptEHR needs **3 files** (one more than HALO — `PATIENTS.csv` is required for demographic conditioning):

1. `PATIENTS.csv` — date of birth and gender
2. `ADMISSIONS.csv` — admission timestamps (used to compute age at first admission)
3. `DIAGNOSES_ICD.csv` — ICD-9 diagnosis codes

Files persist across Colab sessions when saved to Google Drive.

In [ ]:
# Check which files exist in the Drive-backed DATA_DIR
required_files = {
    'PATIENTS.csv':      'Patient demographics (DOB, gender)',
    'ADMISSIONS.csv':    'Admission records (timestamps)',
    'DIAGNOSES_ICD.csv': 'ICD-9 diagnosis codes',
}
existing = {f: os.path.exists(f'{DATA_DIR}/{f}') for f in required_files}
missing  = [f for f, ok in existing.items() if not ok]

if not missing:
    # All files already in Drive — no upload needed
    print("✓ All MIMIC-III files found in Drive (no upload needed):")
    for fname in required_files:
        size_mb = os.path.getsize(f'{DATA_DIR}/{fname}') / 1024 / 1024
        print(f"    {fname}  ({size_mb:.1f} MB)")
    print(f"\nFiles are reused from: {DATA_DIR}")
    print("To force re-upload, delete files from that folder and re-run this cell.")
else:
    print("MIMIC-III file status:")
    for fname, desc in required_files.items():
        mark = "✓" if existing[fname] else "✗ MISSING"
        print(f"  {mark}  {fname} — {desc}")

    print(f"\nUploading {len(missing)} missing file(s)...")
    uploaded = files.upload()

    # Normalize filenames — Colab renames duplicates as "ADMISSIONS (1).csv".
    # Match each upload to the required file it belongs to, then copy with
    # the canonical name so subsequent runs find the file in Drive.
    for uploaded_name, data in uploaded.items():
        matched = None
        for req in required_files:
            base = req.replace('.csv', '')
            if base in uploaded_name and uploaded_name.endswith('.csv'):
                matched = req
                break
        if matched:
            tmp = f'/content/{uploaded_name}'
            with open(tmp, 'wb') as f:
                f.write(data)
            dest = f'{DATA_DIR}/{matched}'
            shutil.copy(tmp, dest)
            size_mb = os.path.getsize(dest) / 1024 / 1024
            print(f"  ✓ Saved {matched} ({size_mb:.1f} MB) → {dest}")
        else:
            print(f"  ⚠  Unrecognised file: {uploaded_name} (skipped)")

    missing = [f for f in required_files if not os.path.exists(f'{DATA_DIR}/{f}')]
    if missing:
        raise FileNotFoundError(
            f"Still missing: {missing}. Please re-run this cell to upload them."
        )
    print("\n✓ All 3 MIMIC-III files present.")

In [ ]:
print("Validating MIMIC-III files...")

_patients = pd.read_csv(f'{DATA_DIR}/PATIENTS.csv')
assert 'SUBJECT_ID' in _patients.columns, "PATIENTS.csv missing SUBJECT_ID"
assert 'GENDER'     in _patients.columns, "PATIENTS.csv missing GENDER"
assert 'DOB'        in _patients.columns, "PATIENTS.csv missing DOB"
print(f"✓ PATIENTS.csv:      {len(_patients):>8,} rows")

_admissions = pd.read_csv(f'{DATA_DIR}/ADMISSIONS.csv')
assert 'SUBJECT_ID' in _admissions.columns, "ADMISSIONS.csv missing SUBJECT_ID"
assert 'HADM_ID'    in _admissions.columns, "ADMISSIONS.csv missing HADM_ID"
print(f"✓ ADMISSIONS.csv:    {len(_admissions):>8,} rows")

_diagnoses = pd.read_csv(f'{DATA_DIR}/DIAGNOSES_ICD.csv')
assert 'ICD9_CODE'  in _diagnoses.columns, "DIAGNOSES_ICD.csv missing ICD9_CODE"
print(f"✓ DIAGNOSES_ICD.csv: {len(_diagnoses):>8,} rows")

del _patients, _admissions, _diagnoses  # free memory
print("\n✓ All files validated successfully")

---
# 4. Training

**What happens during training:**

1. **Dataset loading**: PyHealth reads MIMIC-III and creates one sample per patient (nested visit sequences + demographics: age at first admission, gender).
2. **Tokenization**: Each ICD-9 code is mapped to a unique BART token ID. Special tokens mark visit boundaries: `[VISIT_START]`, `[VISIT_END]`, `[SEQ_END]`.
3. **Demographic prompts**: Age and gender are encoded into learned prompt vectors prepended to the BART encoder input — steering the model toward age/gender-appropriate diagnosis patterns.
4. **Fine-tuning**: HuggingFace Trainer fine-tunes the BART Seq2Seq model to predict the next token conditioned on the demographic prompts.
5. **Checkpoint**: Saved to `{CHECKPOINT_DIR}/checkpoint.pt` after training.

The `WARMUP_STEPS` ramp up the learning rate gradually during early training, preventing catastrophic forgetting of BART's pretrained sequence modeling capabilities.

In [ ]:
# Set all random seeds before any stochastic operation
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
print(f"✓ Random seed set to {SEED}")

from pyhealth.datasets import MIMIC3Dataset, split_by_patient
from pyhealth.tasks import promptehr_generation_mimic3_fn
from pyhealth.models import PromptEHR

print("\nLoading MIMIC-III dataset (this may take a few minutes)...")
dataset = MIMIC3Dataset(
    root=DATA_DIR,
    tables=["patients", "admissions", "diagnoses_icd"],
)
print(f"Loaded {len(dataset.unique_patient_ids):,} patients")

print("Applying PromptEHR generation task...")
sample_dataset = dataset.set_task(promptehr_generation_mimic3_fn)
print(f"Eligible patients (≥2 visits with ICD-9 codes): {len(sample_dataset):,}")

train_dataset, val_dataset, _ = split_by_patient(sample_dataset, [0.8, 0.1, 0.1])
print(f"\nSplit: {len(train_dataset):,} train / {len(val_dataset):,} val patients")

In [ ]:
# Initialize model
print("Initializing PromptEHR model...")
model = PromptEHR(
    dataset=train_dataset,
    n_num_features=1,        # 1 continuous demographic feature: age
    cat_cardinalities=[2],   # 1 categorical feature: gender (binary: 0=male, 1=female)
    d_hidden=D_HIDDEN,
    prompt_length=PROMPT_LENGTH,
    bart_config_name=BART_CONFIG_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    warmup_steps=WARMUP_STEPS,
    max_seq_length=MAX_SEQ_LENGTH,
    save_dir=CHECKPOINT_DIR,
)

n_special = 7  # PAD, BOS, EOS, UNK, VISIT_START, VISIT_END, SEQ_END
n_codes   = model._vocab.total_size - n_special
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ PromptEHR initialized")
print(f"  Vocabulary: {model._vocab.total_size} tokens "
      f"({n_codes} ICD-9 codes + {n_special} special tokens)")
print(f"  Parameters: {total_params:,}")

# Train
print("\nStarting training...")
print("HuggingFace Trainer will print step-by-step progress below.")
print("=" * 60)

model.train_model(train_dataset, val_dataset=val_dataset)

print("=" * 60)
print("✓ Training complete!")
print(f"  Checkpoint: {CHECKPOINT_DIR}/checkpoint.pt")

---
# 5. Generation

**How generation works:**

1. **Demographic sampling**: For each synthetic patient, `synthesize_dataset` draws an `(age, gender)` pair from `model._demo_pool` — the real training population. This ensures the synthetic cohort's demographic profile mirrors MIMIC-III.
2. **Prompt conditioning**: The sampled demographics are encoded into prompt vectors and prepended to the BART encoder input.
3. **Autoregressive decoding**: BART generates tokens one at a time. Special tokens `[VISIT_START]` and `[VISIT_END]` structure the output into visits; `[SEQ_END]` ends the patient sequence.
4. **Decoding**: Token IDs are mapped back to ICD-9 code strings.

`RANDOM_SAMPLING = True` (default): nucleus sampling — diverse, realistic output.  
`RANDOM_SAMPLING = False`: greedy decoding — deterministic, may repeat common patterns.

In [ ]:
print(f"Generating {N_SYNTHETIC_SAMPLES:,} synthetic patients...")
print(f"  Sampling: {'nucleus (random)' if RANDOM_SAMPLING else 'greedy'}"
      + (f", temperature={TEMPERATURE}, top_p={TOP_P}" if RANDOM_SAMPLING else ""))
print("(This may take several minutes...)")

synthetic = model.synthesize_dataset(
    num_samples=N_SYNTHETIC_SAMPLES,
    random_sampling=RANDOM_SAMPLING,
)

print(f"\n✓ Generated {len(synthetic):,} synthetic patients")

# Preview
_preview = []
for p in synthetic[:10]:
    _v0 = p["visits"][0] if p["visits"] else []
    _sample = ", ".join(_v0[:4]) + ("..." if len(_v0) > 4 else "")
    _preview.append({
        "patient_id":       p["patient_id"],
        "n_visits":         len(p["visits"]),
        "total_codes":      sum(len(v) for v in p["visits"]),
        "first_visit_codes": _sample or "(empty)",
    })
display(pd.DataFrame(_preview))

In [ ]:
# Save as CSV (flat SUBJECT_ID, VISIT_NUM, ICD9_CODE — matches MIMIC-III output schema)
_rows = []
for p in synthetic:
    for _vnum, _visit in enumerate(p["visits"], 1):
        for _code in _visit:
            _rows.append({"SUBJECT_ID": p["patient_id"],
                          "VISIT_NUM":  _vnum,
                          "ICD9_CODE":  _code})
df_synthetic = pd.DataFrame(_rows)
csv_path = f'{OUTPUT_DIR}/synthetic_patients.csv'
df_synthetic.to_csv(csv_path, index=False)
print(f"✓ {len(df_synthetic):,} records → {csv_path}")
print(f"  Columns: SUBJECT_ID, VISIT_NUM, ICD9_CODE")
print("\nSample rows:")
display(df_synthetic.head(8))

---
# 6. Results & Download

In [ ]:
# Validate generated data
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# Check 1: Patient IDs
unique_patients = df_synthetic['SUBJECT_ID'].nunique()
print(f"\nUnique patients: {unique_patients} out of {N_SYNTHETIC_SAMPLES} requested")

# Check 2: No empty values
empty_subjects = df_synthetic['SUBJECT_ID'].isna().sum()
empty_visits = df_synthetic['VISIT_NUM'].isna().sum()
empty_codes = df_synthetic['ICD9_CODE'].isna().sum()

print(f"\nEmpty values check:")
print(f"  Subject IDs: {empty_subjects} (should be 0)")
print(f"  Visit numbers: {empty_visits} (should be 0)")
print(f"  ICD9 codes: {empty_codes} (should be 0)")
assert empty_subjects == 0 and empty_visits == 0 and empty_codes == 0, "Found empty values!"

# Check 3: Distribution statistics
codes_per_patient = df_synthetic.groupby('SUBJECT_ID').size()
print(f"\nCodes per patient:")
print(f"  Min: {codes_per_patient.min()}")
print(f"  Max: {codes_per_patient.max()}")
print(f"  Mean: {codes_per_patient.mean():.2f}")
print(f"  Median: {codes_per_patient.median():.2f}")

visits_per_patient = df_synthetic.groupby('SUBJECT_ID')['VISIT_NUM'].max()
print(f"\nVisits per patient:")
print(f"  Min: {visits_per_patient.min()}")
print(f"  Max: {visits_per_patient.max()}")
print(f"  Mean: {visits_per_patient.mean():.2f}")
print(f"  Median: {visits_per_patient.median():.2f}")

print("\n" + "=" * 60)
print("ALL QUALITY CHECKS PASSED")
print("=" * 60)

In [ ]:
# Download CSV file
print("=" * 60)
print("DOWNLOAD SYNTHETIC DATA")
print("=" * 60)

print(f"\nYour synthetic data is ready:")
print(f"  File: synthetic_patients.csv")
print(f"  Patients: {unique_patients:,}")
print(f"  Total records: {len(df_synthetic):,}")
print(f"  Size: {os.path.getsize(csv_path) / (1024*1024):.2f} MB")

print(f"\nDownloading file to your computer...")
files.download(csv_path)

print(f"\nDownload started!")
print(f"\nFile also saved in Google Drive:")
print(f"  {csv_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CHECKPOINT RESUME — Run this cell instead of Section 4 if you already trained
# ─────────────────────────────────────────────────────────────────────────────
# Uncomment everything below to load an existing checkpoint, then skip to Section 5.

# from pyhealth.datasets import MIMIC3Dataset, split_by_patient
# from pyhealth.tasks import promptehr_generation_mimic3_fn
# from pyhealth.models import PromptEHR
#
# dataset = MIMIC3Dataset(
#     root=DATA_DIR,
#     tables=["patients", "admissions", "diagnoses_icd"],
# )
# sample_dataset = dataset.set_task(promptehr_generation_mimic3_fn)
# train_dataset, val_dataset, _ = split_by_patient(sample_dataset, [0.8, 0.1, 0.1])
#
# model = PromptEHR(
#     dataset=train_dataset,
#     n_num_features=1, cat_cardinalities=[2],
#     d_hidden=D_HIDDEN, prompt_length=PROMPT_LENGTH,
#     bart_config_name=BART_CONFIG_NAME,
#     epochs=EPOCHS, batch_size=BATCH_SIZE,
#     lr=LR, warmup_steps=WARMUP_STEPS,
#     max_seq_length=MAX_SEQ_LENGTH,
#     save_dir=CHECKPOINT_DIR,
# )
# ckpt = f'{CHECKPOINT_DIR}/checkpoint.pt'
# model.load_model(ckpt)
# print(f"✓ Loaded checkpoint from {ckpt}. Proceed to Section 5.")

print("(Resume template — uncomment the lines above to use)")

---
## Congratulations!

You've successfully:
1. Trained a PromptEHR model conditioned on patient demographics
2. Generated synthetic patients whose age/gender distribution mirrors MIMIC-III
3. Validated the synthetic data quality
4. Downloaded the CSV file

## Next Steps

**Use your synthetic data:**
- Train readmission/mortality/LoS prediction models on synthetic data
- Evaluate fairness across demographic subgroups
- Share synthetic patients without privacy concerns

**Generate more samples:**
- Change `N_SYNTHETIC_SAMPLES` and re-run Section 5
- No need to retrain — the checkpoint is saved!

**Production training:**
- For publication-quality results, set `EPOCHS = 20` and `N_SYNTHETIC_SAMPLES = 10000`
- Consider using Colab Pro or a dedicated GPU cluster
- See `examples/slurm/` for cluster usage

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `AssertionError: transformers>=4.48.3 required` | Old transformers installed | `pip install transformers --upgrade` |
| Empty patients in output | Undertrained model | Increase `EPOCHS` or raise `TEMPERATURE` to `1.0` |
| Training loss not decreasing after 2+ epochs | LR too high | Try `LR = 5e-6` and `WARMUP_STEPS = 500` |
| Out of memory (OOM) | Batch too large | Reduce `BATCH_SIZE = 8` |
| Very slow training | No GPU | Runtime → Change runtime type → T4 GPU |
| Synthetic codes all the same | Temperature too low | Try `TEMPERATURE = 1.0`, `RANDOM_SAMPLING = True` |

## Reference

Wang, Y., et al. "PromptEHR: Conditional Electronic Healthcare Records Generation with Prompt Learning." *EMNLP 2023*. https://arxiv.org/abs/2211.01761

---
_Notebook for PyHealth 2.0 · Branch: `promptehr-pr-integration` · jalengg/PyHealth_